In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
images_dir = '/content/drive/MyDrive/NMAI/results'
mapping_dir = '/content/drive/MyDrive/NMAI/results/labels.csv'

In [ ]:
# @title
import os
import pandas as pd

# Count images in images_dir
image_count = 0
image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff') # Add more if needed

if os.path.exists(images_dir) and os.path.isdir(images_dir):
    for filename in os.listdir(images_dir):
        if filename.lower().endswith(image_extensions):
            image_count += 1
    print(f"Number of images in '{images_dir}': {image_count}")
else:
    print(f"Error: Image directory '{images_dir}' not found or is not a directory.")

# Load mapping.csv and print head
if os.path.exists(mapping_dir):
    try:
        mapping_df = pd.read_csv(mapping_dir)
        print("\nHead of mapping.csv:")
        print(mapping_df.head())
    except Exception as e:
        print(f"Error loading mapping.csv: {e}")
else:
    print(f"Error: Mapping file '{mapping_dir}' not found.")

Number of images in '/content/drive/MyDrive/NMAI/results': 1970

Head of mapping.csv:
  filename         label
0    1.jpg    53-V7 3196
1    2.jpg  29-C1 209.02
2    3.jpg  47-B1 428.72
3    4.jpg  29-AA 434.56
4    5.jpg  60-B8 499.28


In [ ]:
# @title
# Install necessary libraries if not already installed
!pip install transformers accelerate evaluate jiwer

import os
import pandas as pd
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
import evaluate

# Load pre-trained TrOCR model and processor
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-large-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-handwritten")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/635 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-23): 24 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=False)
              (key): Linear(in_features=1024, out_features=1024, bias=False)
              (value): Linear(in_features=1024, out_features=1024, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=1024, out_features=4096, bias=True)
    

In [ ]:
import albumentations as A
import numpy as np
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 0. Chia dữ liệu thành tập train và eval
train_df, eval_df = train_test_split(mapping_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

train_transforms = A.Compose([
    # 1. Biến đổi hình học (Perspective/Rotation)
    A.Perspective(scale=(0.05, 0.1), p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.Affine(shear=(-10, 10), p=0.3),

    # 2. Biến đổi quang học (Brightness/Contrast/Glare)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.OneOf([
        A.RandomShadow(p=1),
        A.RandomSunFlare(p=1), # Đã xóa flare_center_height/width không hợp lệ
    ], p=0.3),

    # 3. Nhiễu và Mờ (Blur/Noise)
    A.OneOf([
        A.MotionBlur(p=1),
        A.GaussianBlur(p=1),
        A.GaussNoise(p=1), # Đã xóa var_limit không hợp lệ ở bản cũ
    ], p=0.4),
])

class LicensePlateDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=32, transform=None):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df['filename'][idx]
        text = self.df['label'][idx]

        image = Image.open(os.path.join(self.root_dir, file_name)).convert("RGB")

        # Áp dụng Augmentation nếu là tập train
        if self.transform:
            image_np = np.array(image)
            augmented = self.transform(image=image_np)
            image = Image.fromarray(augmented['image'])

        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(text,
                                          padding="max_length",
                                          max_length=self.max_target_length).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}

# Khởi tạo lại dataset
train_dataset = LicensePlateDataset(
    root_dir=images_dir,
    df=train_df,
    processor=processor,
    transform=train_transforms
)

eval_dataset = LicensePlateDataset(
    root_dir=images_dir,
    df=eval_df,
    processor=processor
)

print(f"Datasets re-initialized with Data Augmentation for training.")
print(f"Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")

Datasets re-initialized with Data Augmentation for training.
Train samples: 1773, Eval samples: 197


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Fix model config for training
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size

# Set training arguments
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="steps",
    save_strategy="no",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    fp16=True,
    output_dir="./trocr-license-plate",
    logging_steps=10,
    eval_steps=100,
    num_train_epochs=10,
    learning_rate=5e-5,
    report_to="none"
)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=None
)

# Start training
print("Starting fine-tuning...")
trainer.train()

Starting fine-tuning...


Step,Training Loss,Validation Loss
100,0.467275,0.371292
200,0.649974,0.263608
300,0.638936,0.257089
400,0.422731,0.286005
500,0.437416,0.158400
600,0.385543,0.195080
700,0.390645,0.119964
800,0.307030,0.118722
900,0.565553,0.143367
1000,0.682606,0.121022


TrainOutput(global_step=4440, training_loss=0.26456209966862526, metrics={'train_runtime': 3058.0384, 'train_samples_per_second': 5.798, 'train_steps_per_second': 1.452, 'total_flos': 2.6244868960042353e+19, 'train_loss': 0.26456209966862526, 'epoch': 10.0})

In [ ]:
# @title
from tqdm.auto import tqdm
import evaluate

# Load metrics
cer_metric = evaluate.load("cer")

def evaluate_model(model, dataset, processor, device):
    model.eval()
    predictions = []
    references = []

    print(f"Evaluating on {len(dataset)} samples...")

    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        pixel_values = batch["pixel_values"].unsqueeze(0).to(device)

        # Generate transcription
        with torch.no_grad():
            generated_ids = model.generate(pixel_values)

        generated_text = processor.decode(generated_ids[0], skip_special_tokens=True)

        # Get reference text (removing -100 padding tokens)
        labels = batch["labels"]
        labels = [l for l in labels if l != -100]
        reference_text = processor.decode(labels, skip_special_tokens=True)

        predictions.append(generated_text)
        references.append(reference_text)

    # Compute CER
    cer = cer_metric.compute(predictions=predictions, references=references)

    # Compute Exact Match
    exact_matches = sum([1 for pred, ref in zip(predictions, references) if pred.strip() == ref.strip()])
    accuracy = exact_matches / len(predictions)

    print(f"\n--- Final Evaluation Results ---")
    print(f"Character Error Rate (CER): {cer:.4f}")
    print(f"Exact Match Accuracy: {accuracy:.4f} ({exact_matches}/{len(predictions)})")

    # Display some examples
    print("\n--- Sample Results ---")
    for i in range(min(5, len(predictions))):
        print(f"Pred: '{predictions[i]}' | Actual: '{references[i]}'")

# Run evaluation
evaluate_model(model, eval_dataset, processor, device)

Evaluating on 197 samples...


  0%|          | 0/197 [00:00<?, ?it/s]


--- Final Evaluation Results ---
Character Error Rate (CER): 0.0067
Exact Match Accuracy: 0.9239 (182/197)

--- Sample Results ---
Pred: '59-L2 313.95' | Actual: '59-L2 313.95'
Pred: '98-N5 1102' | Actual: '98-N5 1102'
Pred: '59-S2 165.55' | Actual: '59-S2 165.55'
Pred: '56-P1 6734' | Actual: '56-P1 6734'
Pred: '59-S2 470.61' | Actual: '59-S2 470.61'


In [ ]:
# Define the save path in Google Drive
save_path = '/content/drive/MyDrive/NMAI/trocr_license_plate_finetuned_2'

# Create directory if it doesn't exist
import os
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Save the model and processor
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f"Model and processor saved successfully to: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor saved successfully to: /content/drive/MyDrive/NMAI/trocr_license_plate_finetuned_2
